In [21]:
# Load necessary packages

import pandas as pd
import numpy as np
from pathlib import Path

RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")

In [22]:
# Load Raw Data

q1 = pd.read_csv(RAW_DATA_DIR / "Divvy_Trips_2019_Q1.csv")
q2 = pd.read_csv(RAW_DATA_DIR / "Divvy_Trips_2019_Q2.csv")
q3 = pd.read_csv(RAW_DATA_DIR / "Divvy_Trips_2019_Q3.csv")
q4 = pd.read_csv(RAW_DATA_DIR / "Divvy_Trips_2019_Q4.csv")

In [23]:
# Reformat incorrect columns in q2

q2.columns = [
    "trip_id", "start_time", "end_time", "bikeid", "tripduration",
    "from_station_id", "from_station_name", "to_station_id",
    "to_station_name", "usertype", "gender", "birthyear"
]

In [24]:
# Combine datasets

df = pd.concat([q1, q2, q3, q4], ignore_index=True)

In [25]:
# Adjust names of ridertype and break down time fields for analysis

df["rider_type"] = df["usertype"].replace({
    "Customer": "Casual Rider",
    "Subscriber": "Annual Member"
})

df["tripduration"] = (
    df["tripduration"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["tripduration"] = pd.to_numeric(df["tripduration"], errors="coerce")
df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce")
df["end_time"] = pd.to_datetime(df["end_time"])
df["trip_duration_minutes"] = df["tripduration"] / 60
df["day_of_week"] = df["start_time"].dt.day_name()
df["month"] = df["start_time"].dt.month
df["hour"] = df["start_time"].dt.hour

In [26]:
# Break down by seasons

df["season"] = np.select(
    [
        df["month"].isin([12, 1, 2]),
        df["month"].isin([3, 4, 5]),
        df["month"].isin([6, 7, 8]),
        df["month"].isin([9, 10, 11])
    ],
    ["Winter", "Spring", "Summer", "Fall"],
    default="Unknown"
)

In [27]:
df.to_csv(PROCESSED_DATA_DIR / "cyclistic_2019_cleaned.csv", index=False)